# MagicFinance — Module D: Forecast Engine

**Goal**: Use Qwen 4B as a calibrated prediction market for binary financial events.  
Generates per-stock events from Module A signals + macro events from NewsAPI.  
Stores forecast history in Qdrant for accuracy tracking over time.

**Pipeline**:
1. Load high-confidence signals from Module A (Qdrant)
2. Generate per-stock binary events from each ticker signal
3. Fetch macro news and generate macro binary events
4. Forecast each event with Qwen 4B → probability + reasoning
5. Store forecasts in Qdrant
6. Fire Slack alerts for `forecast_probability >= 0.70`
7. Backtest: accuracy of past forecasts (if resolved data exists)

**Requires**: Module A must have run first (signals in Qdrant)

**Output**: Forecasts stored in `magicfinance_forecast_history`

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import requests
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import display

load_dotenv(dotenv_path='../.env')

from magicfinance import config
from magicfinance.llm_client import forecast_binary_event, generate_events_from_signal, check_ollama_health
from magicfinance import qdrant_client as qc
from magicfinance.slack_client import alert_strong_forecast

print(f'Model: {config.MODEL_4B}')
print(f'Forecast threshold: {config.FORECAST_THRESHOLD}')

## 1. Health Checks

In [ ]:
ollama_ok = check_ollama_health()
print(f'Ollama: {"✅" if ollama_ok else "❌"}')

try:
    qc.ensure_collections()
    signals = qc.get_investable_signals()
    print(f'Qdrant: ✅ — {len(signals)} investable signals from Module A')
    if not signals:
        print('⚠️  No investable signals found. Run notebook_a.ipynb first.')
except Exception as e:
    print(f'Qdrant: ❌ — {e}')

## 2. Fetch Macro Events from NewsAPI

In [ ]:
def fetch_macro_news(queries: list, api_key: str, articles_per_query: int = 5) -> list[dict]:
    """
    Fetch recent financial news headlines for macro event generation.
    
    Uses NewsAPI.org free tier (100 req/day, 1 month history).
    Returns list of article dicts with title, description, publishedAt.
    """
    if not api_key:
        print('⚠️  NEWSAPI_KEY not set in .env — skipping macro news')
        return []
    
    articles = []
    for query in queries:
        try:
            resp = requests.get(
                f'{config.NEWSAPI_BASE_URL}/everything',
                params={
                    'q': query,
                    'language': 'en',
                    'sortBy': 'publishedAt',
                    'pageSize': articles_per_query,
                    'apiKey': api_key,
                },
                timeout=10,
            )
            data = resp.json()
            for a in data.get('articles', []):
                articles.append({
                    'query': query,
                    'title': a.get('title', ''),
                    'description': a.get('description', ''),
                    'published_at': a.get('publishedAt', ''),
                    'source': a.get('source', {}).get('name', ''),
                })
        except Exception as e:
            print(f'  NewsAPI error for "{query}": {e}')
    
    print(f'Fetched {len(articles)} macro news articles')
    return articles


NEWSAPI_KEY = os.environ.get('NEWSAPI_KEY', '')
macro_articles = fetch_macro_news(config.NEWSAPI_QUERIES, NEWSAPI_KEY, articles_per_query=config.NEWSAPI_ARTICLE_LIMIT)

# Generate macro binary events from news context
MACRO_EVENTS = [
    {
        'event': 'Will the Federal Reserve cut interest rates at the next FOMC meeting?',
        'context': ' '.join(a['title'] for a in macro_articles if 'Federal' in a.get('query', '') or 'rate' in a.get('title', '').lower()),
        'is_macro_event': True,
        'ticker': None,
    },
    {
        'event': 'Will US CPI inflation fall below 3% in the next monthly report?',
        'context': ' '.join(a['title'] for a in macro_articles if 'CPI' in a.get('query', '') or 'inflation' in a.get('title', '').lower()),
        'is_macro_event': True,
        'ticker': None,
    },
    {
        'event': 'Will the S&P 500 close higher 1 month from today?',
        'context': ' '.join(a['title'] for a in macro_articles if 'S&P' in a.get('query', '') or 'market' in a.get('title', '').lower()),
        'is_macro_event': True,
        'ticker': None,
    },
]

print(f'Macro events to forecast: {len(MACRO_EVENTS)}')

## 3. Generate Per-Stock Events from Module A Signals

In [ ]:
signals = qc.get_investable_signals(min_confidence=0.60)
print(f'Processing {len(signals)} investable signals...')

per_stock_events = []

for signal in signals[:15]:  # cap at 15 to avoid too many LLM calls
    ticker = signal.get('ticker', '')
    if not ticker:
        continue
    
    # Auto-generate relevant binary questions for this ticker
    try:
        generated = generate_events_from_signal(signal)
        for ev in generated:
            per_stock_events.append({
                'event': ev['event'],
                'context': f"Reddit thesis: {signal.get('explanation', {}).get('thesis_score', '')} Confidence: {signal.get('confidence_level', 0):.2f}",
                'is_macro_event': False,
                'ticker': ticker,
            })
        print(f'  {ticker}: generated {len(generated)} events')
    except Exception as e:
        print(f'  {ticker}: event generation failed — {e}')

print(f'\nPer-stock events generated: {len(per_stock_events)}')

## 4. Forecast All Events with Qwen 4B

In [ ]:
all_events = per_stock_events + MACRO_EVENTS
print(f'Forecasting {len(all_events)} events with {config.MODEL_4B}...')

forecasts = []

for i, ev in enumerate(all_events):
    try:
        result = forecast_binary_event(ev['event'], ev['context'])
        forecast = {
            **result,
            'ticker': ev.get('ticker'),
            'is_macro_event': ev.get('is_macro_event', False),
            'signal_timestamp': datetime.utcnow().isoformat(),
            'resolved': False,  # Will be updated when outcome is known
            'actual_outcome': None,
        }
        forecasts.append(forecast)
        
        tag = '🌍 MACRO' if ev.get('is_macro_event') else f'📈 {ev.get("ticker", "")}'
        print(f'  [{i+1}/{len(all_events)}] {tag} | p={result.get("forecast_probability", 0):.0%} | {ev["event"][:60]}')
    except Exception as e:
        print(f'  [{i+1}/{len(all_events)}] ERROR: {e}')

print(f'\nForecasts generated: {len(forecasts)}')

## 5. Store in Qdrant + Fire Slack Alerts

In [ ]:
alerts_fired = 0

for forecast in forecasts:
    # Store all forecasts in Qdrant (even low probability — for calibration tracking)
    qc.upsert_forecast(forecast)
    
    # Fire Slack alert for strong forecasts
    if forecast.get('forecast_probability', 0) >= config.FORECAST_THRESHOLD:
        alert_strong_forecast(forecast)
        alerts_fired += 1

print(f'Stored {len(forecasts)} forecasts in Qdrant')
print(f'Slack alerts fired: {alerts_fired} (threshold: probability >= {config.FORECAST_THRESHOLD})')

## 6. Forecast Distribution Analysis

In [ ]:
all_forecasts = qc.get_forecast_history()
df_f = pd.DataFrame(all_forecasts)

print(f'Total forecasts in Qdrant: {len(df_f)}')
print(f'Stock events: {(~df_f.get("is_macro_event", pd.Series(False))).sum()}')
print(f'Macro events: {df_f.get("is_macro_event", pd.Series(False)).sum()}')
print(f'High probability (>={config.FORECAST_THRESHOLD}): {(df_f["forecast_probability"] >= config.FORECAST_THRESHOLD).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('MagicFinance — Module D: Forecast Distribution', fontsize=14, fontweight='bold')

# Probability distribution
axes[0].hist(df_f['forecast_probability'], bins=20, color='mediumpurple', edgecolor='white')
axes[0].axvline(config.FORECAST_THRESHOLD, color='red', linestyle='--', label=f'Alert threshold ({config.FORECAST_THRESHOLD})')
axes[0].set_title('Forecast Probability Distribution')
axes[0].set_xlabel('Probability')
axes[0].legend()

# Top high-probability forecasts
top = df_f.nlargest(10, 'forecast_probability')[['ticker', 'forecast_probability', 'is_macro_event']]
top['label'] = top.apply(lambda r: f'🌍{r["event"][:30] if "event" in r else r["ticker"]}' if r['is_macro_event'] else f'${r["ticker"]}', axis=1)
axes[1].barh(range(len(top)), top['forecast_probability'], color='mediumpurple')
axes[1].set_yticks(range(len(top)))
axes[1].set_yticklabels(df_f.nlargest(10, 'forecast_probability').apply(
    lambda r: ('MACRO: ' if r.get('is_macro_event') else f'${r.get("ticker","?")} ') + r.get('event','')[:40], axis=1
).values)
axes[1].set_title('Top 10 Forecasts by Probability')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig('../data/module_d_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Accuracy Backtest (Resolved Forecasts)

In [ ]:
resolved = qc.get_resolved_forecasts()

if not resolved:
    print('No resolved forecasts yet — accuracy backtest will be available once forecasts are resolved.')
    print('To resolve a forecast: update it in Qdrant with resolved=True and actual_outcome=True/False')
else:
    df_r = pd.DataFrame(resolved)
    df_r['correct'] = df_r['prediction'].map({'yes': True, 'no': False}) == df_r['actual_outcome']
    
    # Accuracy by confidence bucket
    df_r['confidence_bucket'] = pd.cut(df_r['confidence_level'], bins=[0, 0.5, 0.7, 1.0], labels=['Low', 'Medium', 'High'])
    accuracy_by_conf = df_r.groupby('confidence_bucket')['correct'].agg(['mean','count'])
    
    print('Forecast Accuracy by Confidence Level:')
    print(accuracy_by_conf.rename(columns={'mean': 'accuracy', 'count': 'n_forecasts'}).to_string())
    print(f'\nOverall accuracy: {df_r["correct"].mean():.0%} ({len(df_r)} forecasts)')

print('\n✅ Module D complete. Run notebook_e.ipynb next.')